In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(parent_dir)

In [2]:
# Importing packages and modules (Old)
from openpyxl import Workbook, load_workbook
from datetime import datetime
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
import numpy as np
import gymnasium as gym
from itertools import product
from tqdm import tqdm
from RL4CRN.iocrns.mass_action_iocrn import MassActionIOCRN
from RL4CRN.environments.environment import Environment
from RL4CRN.environments.parallel_environments import ParallelEnvironments
from RL4CRN.environments.serial_environments import SerialEnvironments
from RL4CRN.agents.reinforce_agent import REINFORCEAgent
from RL4CRN.policies.add_reaction_by_index import AddReactionByIndex
from RL4CRN.env2agent_interface.test_observer import TestObserver
from RL4CRN.env2agent_interface.test_tensorizer import TestTensorizer
from RL4CRN.agent2env_interface.Identity_actuator import IdentityActuator
from RL4CRN.agent2env_interface.mass_action_iocrn_stepper import MassActionIOCRNStepper
from RL4CRN.rewards.deterministic import dynamic_tracking_error

# Importing packages and modules (New)
from openpyxl import Workbook, load_workbook
from datetime import datetime
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
import numpy as np
from itertools import product
from tqdm import tqdm
from RL4CRN.iocrns.reactions import MassAction
from RL4CRN.iocrns.iocrn import IOCRN
from RL4CRN.utils.ic import IC
from RL4CRN.iocrns.reaction_library import construct_mass_action_library
from RL4CRN.environments.environment import Environment
from RL4CRN.environments.parallel_environments import ParallelEnvironments
from RL4CRN.environments.serial_environments import SerialEnvironments
from RL4CRN.agents.reinforce_agent import REINFORCEAgent
from RL4CRN.policies.add_reaction_by_index import AddReactionByIndex
from RL4CRN.rewards.deterministic import dynamic_tracking_error
from RL4CRN.env2agent_interface.explicit_observer import ExplicitObserver
from RL4CRN.env2agent_interface.explicit_tensorizer import ExplicitTensorizer
from RL4CRN.agent2env_interface.library_actuator import LibraryActuator


# from RL4CRN.policies.test_policy import TestPolicy
# from RL4CRN.agent2env_interface.Identity_actuator import TestActuator
# from RL4CRN.agents.test_agent import TestAgent
from RL4CRN.agent2env_interface.iocrn_stepper import IOCRNStepper

In [3]:
# Construct the template CRN (Old)
species_labels = ['X_1', 'Z_1', 'Z_2']
inputs_labels = ['u_1', 'u_2']
S_R = np.array([[0, 1], [0, 0], [0, 0]], dtype=np.int8)
S_P = np.array([[0, 0], [1, 0], [0, 0]], dtype=np.int8)
c = np.array([1, 1], dtype=np.float32)
S_I = np.array([[1, 0], [0, 1]], dtype=np.int8)
o = np.array([1], dtype=np.int8)
crn_template = MassActionIOCRN(S_R, S_P, c, S_I, o, species_labels, inputs_labels)
n = len(species_labels)
p = len(inputs_labels)
print('CRN template:')
print(crn_template)

# Construct the template CRN (New)
r1 = MassAction([], ['Z_1'], ['u_1'], [1.])
r2 = MassAction(['X_1'], [], ['u_2'], [1.])
crn_template_new = IOCRN([r1, r2], output_labels=['X_1'])
crn_template_new.compile()
num_inputs_new = crn_template_new.num_inputs
print('CRN template (New):')
print(crn_template_new)

CRN template:
Inputs: ['u_1', 'u_2'] 
Species: ['X_1', 'Z_1', 'Z_2'] 
Output Species: ['X_1'] 
Reaction 0: 0 -> Z_1 ; Rate Constant: 1.0u_1 
Reaction 1: X_1 -> 0 ; Rate Constant: 1.0u_2 

CRN template (New):
Inputs: ['u_1', 'u_2'] 
Species: ['X_1', 'Z_1'] 
Output Species: ['X_1'] 
∅ ----> Z_1;  [MAK(1.0, u_1)]
X_1 ----> ∅;  [MAK(1.0, u_2)]


In [4]:
# Device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
logger = None

Using device: cuda


In [5]:
# Hyperparameters
N_CPUs = os.cpu_count()                             # Number of CPUs          
N = 1*N_CPUs                                        # Number of samples (batch size)    

# Time horizon for the simulation
t_f = 200                                           # Final time for the simulation
N_t = 1000                                          # Number of time steps
time_horizon = np.linspace(0, t_f, N_t, dtype=np.float32)

# Neural network hyperparameters
width = 1024                                        # Width of the neural networks  
depth = 5                                           # Depth of the neural networks 
deep_layer_size = 1024*10                           # Size of the deep layer encoding the CRNs
allow_input_influence = False                       # Allow input influence in the policy
learning_rate = 1e-4                                # Learning rate for the optimizer 
hall_of_fame_size = 10                              # Size of the hall of fame  
entropy_scheduler = {                               # Entropy scheduler parameters
    'entropy_weight': 50.0, 
    'entropy_update_coefficient': 0.75, 
    'entropy_schedule': 5, 
    'minimum_entropy_weight': 50.0
}
risk_scheduler = {                                  # Risk scheduler parameters
    'risk': 0.95, 
    'risk_update': 0.0, 
    'max_risk': 1.0, 
    'risk_schedule': 20
}

# Construct mass action library (New)
num_species = len(species_labels)
library = construct_mass_action_library(species_labels, 2)
crn_template_new.set_library_context(library)

# Construct the IOCRN inputs and initial conditions
nums = [0.5, 1.0, 1.5]
u_list = [np.array(u) for u in product(nums, repeat=p)] # list of input combinations, each input is a numpy array of shape (p,)
x0_list = [np.array([0, 0, 0], dtype=np.float32)] # list of initial conditions, each initial condition is a numpy array of shape (n,)

# Construct the IOCRN initial conditions (New)
ic = IC(species_labels, values = [[0., 0., 0.]]) # Initial conditions object

# Construct the weights for the performance metric
w = np.ones(N_t)
w[(len(w)//5)*4:] = w[(len(w)//5)*4:]*2
w[:(len(w)//5)] = w[:(len(w)//5)]*0.25
w = w[np.newaxis, :]

# Construct the compute reward routine
def compute_reward(state):
   r_list = [np.array([u[0] * state.c[0]]) for u in u_list]
   return dynamic_tracking_error(state, u_list, x0_list, time_horizon, r_list, w, norm=1, LARGE_NUMBER=1e4)

# Construct the compute reward routine (New)
def compute_reward_new(state):
   r_list = [np.array([u[0]]) for u in u_list]
   x0_list_new = ic.get_ic(state)
   return dynamic_tracking_error(state, u_list, x0_list_new, time_horizon, r_list, w, norm=1, LARGE_NUMBER=1e4)

In [6]:
# Construct the interfaces (Old)
crn_0 = crn_template.clone()
M = crn_0.get_reactions_range()
observer = TestObserver(M)
tensorizer = TestTensorizer(device=device)
actuator = IdentityActuator()
stepper = MassActionIOCRNStepper()

# Construct the interfaces (New)
crn_0_new = crn_template_new.clone()
observer_new = ExplicitObserver(library)
tensorizer_new = ExplicitTensorizer(device=device)
actuator_new = LibraryActuator(library)
stepper_new = IOCRNStepper()

In [7]:
# Generate random reaction indices
from random import randrange
import matplotlib.pyplot as plt
rand_nums = [randrange(M) for _ in range(3)]

# Generate policy action (Old)
action1 = {"reaction index": rand_nums[0], "parameters": [1.0]}
action2 = {"reaction index": rand_nums[1], "parameters": [0.1]}
action3 = {"reaction index": rand_nums[2], "parameters": [0.5]}

# Generate policy action (Old)
action1_new = {"reaction index": rand_nums[0], "parameters": [1.0]}
action2_new = {"reaction index": rand_nums[1], "parameters": [0.1]}
action3_new = {"reaction index": rand_nums[2], "parameters": [0.5]}

In [8]:
# Generate 5 random integer numbers in the range [0, M)
from random import randrange
import matplotlib.pyplot as plt
rand_nums = [randrange(M) for _ in range(5)]

In [9]:
# Actuator (Old)
action1 = actuator.actuate(action1)
action2 = actuator.actuate(action2)
action3 = actuator.actuate(action3)
print(action1)
print(action2)
print(action3)

# Actuator (New)
action1_new = actuator_new.actuate(action1_new)
action2_new = actuator_new.actuate(action2_new)
action3_new = actuator_new.actuate(action3_new)
print(action1_new)
print(action2_new)
print(action3_new)

{'reaction index': 70, 'rate constant': 1.0}
{'reaction index': 75, 'rate constant': 0.1}
{'reaction index': 59, 'rate constant': 0.5}
Z_1 + Z_1 ----> X_1 + Z_2;  [MAK(1.0)]
Z_1 + Z_2 ----> Z_1;  [MAK(0.1)]
X_1 + Z_2 ----> X_1 + X_1;  [MAK(0.5)]


In [10]:
crn_0 = crn_template.clone()
crn_0_new = crn_template_new.clone()

# Stepper (Old)
stepper.step(crn_0, action1)
t, x_list, y_list, task_info = crn_0.transient_response(u_list, x0_list, time_horizon)
# fig, ax = crn_0.plot_transient_response(alpha = 1)

# Stepper (New)
stepper_new.step(crn_0_new, action1_new)
x0_list_new = ic.get_ic(crn_0_new)
t, x_list, y_list, task_info = crn_0_new.transient_response(u_list, x0_list_new, time_horizon)
# fig, ax = crn_0_new.plot_transient_response(alpha = 1)

# Rewards
print(compute_reward(crn_0)[0])
print(compute_reward_new(crn_0_new)[0])

0.4100704792312702
0.4100704792312591


In [11]:
# Stepper (Old)
stepper.step(crn_0, action2)
t, x_list, y_list, task_info = crn_0.transient_response(u_list, x0_list, time_horizon)
# fig, ax = crn_0.plot_transient_response(alpha = 1)

# Stepper (New)
stepper_new.step(crn_0_new, action2_new)
x0_list_new = ic.get_ic(crn_0_new)
t, x_list, y_list, task_info = crn_0_new.transient_response(u_list, x0_list_new, time_horizon)
# fig, ax = crn_0_new.plot_transient_response(alpha = 1)

# Rewards
print(compute_reward(crn_0)[0])
print(compute_reward_new(crn_0_new)[0])

0.4100715451165333
0.41007154511653543


In [12]:
# Stepper (Old)
stepper.step(crn_0, action3)
t, x_list, y_list, task_info = crn_0.transient_response(u_list, x0_list, time_horizon)
# fig, ax = crn_0.plot_transient_response(alpha = 1)

# Stepper (New)
stepper_new.step(crn_0_new, action3_new)
x0_list_new = ic.get_ic(crn_0_new)
t, x_list, y_list, task_info = crn_0_new.transient_response(u_list, x0_list_new, time_horizon)
# fig, ax = crn_0_new.plot_transient_response(alpha = 1)

# Rewards
print(compute_reward(crn_0)[0])
print(compute_reward_new(crn_0_new)[0])

0.4880875230490604
0.48808752304906033


In [13]:
# Observer (Old)
obs = observer.observe(crn_0)
obs = tensorizer.tensorize(obs)

# Observer (New)
obs_new = observer_new.observe(crn_0_new)
obs_new = tensorizer_new.tensorize(obs_new)

print((obs == obs_new).any())

tensor(True, device='cuda:0')


In [14]:
# Policy 
encoder_attributes = {"hidden_size": width, "num_layers": depth}
structure_head_attributes = {"hidden_size": width, "num_layers": depth}
rate_head_attributes = {"hidden_size": width, "num_layers": depth}
input_influence_head_attributes = {"hidden_size": width, "num_layers": depth}
masks = {"continuous": np.ones((M, 1)), "discrete": None, "logit": None}
policy = AddReactionByIndex(M, M, p, encoder_attributes, deep_layer_size, structure_head_attributes, rate_head_attributes, input_influence_head_attributes, allow_input_influence=False, masks=masks, device=device)

In [15]:
import random
# 1) Seed everything
seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)

# 2) Determinism knobs (CUDA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
# Optional but stricter (PyTorch >=1.12)
# torch.use_deterministic_algorithms(True)
# For matmul determinism (PyTorch >=1.12 on CUDA):
# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"  # set before importing torch

# 3) Disable stochastic layers
policy.eval()

policy_action, logP, entropy = policy(obs.unsqueeze(0))
action = actuator.actuate(policy_action[0])
stepper.step(crn_0, action)
crn_0.reset()
reward = compute_reward(crn_0)
reward = reward[0]
loss = reward * logP - entropy
loss.backward()
nn_params = list(policy.parameters())


seed = 42
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)


policy_action_new, logP_new, entropy_new = policy(obs_new.unsqueeze(0))
action_new = actuator_new.actuate(policy_action_new[0])
stepper_new.step(crn_0_new, action_new)
crn_0_new.reset()
reward_new = compute_reward_new(crn_0_new)
reward_new = reward_new[0]
loss_new = reward_new * logP_new - entropy_new
loss_new.backward()
nn_params_new = list(policy.parameters())
print((nn_params == nn_params_new))

True
